In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 500

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.05
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.05,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.3855, -1.2158,  1.2917,  0.1474, -1.0304,  0.1415, -0.0089,  1.9538,
         -0.5556,  0.8849, -0.9232, -0.1073]], device='cuda:0')
Scaled actions :  tensor([[-0.3855, -1.2158,  1.2917,  0.1474, -1.0304,  0.1415, -0.0089,  1.9538,
         -0.5556,  0.8849, -0.9232, -0.1073]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-9.8905e-19,  1.6241e-08,  7.0808e-20,  5.8642e-10,  1.7468e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4205e-08,
          1.3352e-08, -1.6159e-04,  3.8040e-04, -1.9497e-04, -1.9295e-08,
         -1.5548e-07, -6.5210e-08, -1.6171e-04,  3.8052e-04, -1.9491e-04,
          8.7039e-07,  7.1027e-07,  6.6759e-07, -8.0796e-03,  1.9022e-02,
         -9.7476e-03, -9.6473e-07, -7.7738e-06, -3.2605e-06, -8.0844e-03,
          1.9029e-02, -9.7447e-03,  4.3519e-05, -3.8552e-01, -1.2158e+00,
          1.2917e+00,  1.4738e-01, -1.0304e+00,  1.4148e-01, -8.9155e-03,
          1.9538e+00, -5.5562e-01,  8.8494e-01, -9.2320e-01, -1.0727e-01]],
       device='cuda:0')
torques: [-2.17042206e-16 -9.29340749e-16  4.11052392e-06  5.08141218e-06
  8.74247341e-07  8.00615857e-17  2.46966923e-16 -7.42679812e-16
  4.11052392e-06  5.08141218e-06  8.74247341e-07 -3.36292106e-16]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1734, -3.8651, -0.0930, -0.2174, -1.6282,  0.0671, -0.0131,  3.2293,
          0.1155, -0.6585, -0.6519,  0.1798]], device='cuda:0')
Scaled actions :  tensor([[-0.1734, -3.8651, -0.0930, -0.2174, -1.6282,  0.0671, -0.0131,  3.2293,
          0.1155, -0.6585, -0.6519,  0.1798]], device='cuda:0')
obs :  tensor([[ 1.7218e-01, -9.6014e-02,  7.3608e-02, -2.4371e-03, -3.6383e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -5.6412e-02,
         -1.4978e-02, -3.8809e-04,  3.3986e-02, -1.8435e-01,  5.3027e-02,
         -7.0006e-03,  3.1329e-03, -2.1740e-02,  5.4125e-02, -1.9252e-01,
         -3.6624e-02, -4.1039e-01, -1.4110e-01,  3.4434e-02,  2.1843e-01,
         -1.3451e+00,  3.0485e-01, -1.8016e-02,  1.9870e-02, -1.9071e-01,
          4.3664e-01, -1.4609e+00, -1.4686e-01, -1.7344e-01, -3.8651e+00,
         -9.2984e-02, -2.1744e-01, -1.6282e+00,  6.7061e-02, -1.3116e-02,
          3.2293e+00,  1.1551e-01, -6.5846e-01, -6.5187e-01,  1.7

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 1.7575, -1.3724,  0.8035, -0.0136,  1.6031, -1.1286, -0.0186,  2.2199,
         -0.6779, -1.6517,  0.4091,  0.1298]], device='cuda:0')
Scaled actions :  tensor([[ 1.7575, -1.3724,  0.8035, -0.0136,  1.6031, -1.1286, -0.0186,  2.2199,
         -0.6779, -1.6517,  0.4091,  0.1298]], device='cuda:0')
obs :  tensor([[ 0.0864, -0.0786,  0.1066, -0.0059, -0.0084, -0.9999,  1.0000,  0.0000,
          0.0000, -0.1025, -0.0503,  0.0107,  0.0561, -0.6046,  0.0600, -0.0115,
          0.0164, -0.0364,  0.0981, -0.3565,  0.0269, -0.1641, -0.2049,  0.0621,
          0.0165, -2.1353,  0.0198, -0.0295,  0.1089,  0.0283,  0.0484, -0.6207,
          0.3153,  1.7575, -1.3724,  0.8035, -0.0136,  1.6031, -1.1286, -0.0186,
          2.2199, -0.6779, -1.6517,  0.4091,  0.1298]], device='cuda:0')
torques: [  24.08133085 -200.         -200.         -200.           91.76738617
   -5.92214146   24.95483997  200.          200.         -200.
   30.87384703   -9.97541492]
データ収集:

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.0982, -0.3282, -0.3466,  0.4115,  1.6366,  0.8127,  0.3331,  0.2459,
          0.7743, -1.3308,  0.5884, -0.5400]], device='cuda:0')
Scaled actions :  tensor([[ 0.0982, -0.3282, -0.3466,  0.4115,  1.6366,  0.8127,  0.3331,  0.2459,
          0.7743, -1.3308,  0.5884, -0.5400]], device='cuda:0')
obs :  tensor([[ 0.0694, -0.1251, -0.6780, -0.0098, -0.0114, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0540, -0.0883,  0.0395,  0.0592, -0.8303, -0.0748, -0.0113,
          0.0487, -0.0340,  0.0879, -0.2697,  0.0750,  0.5572, -0.1553,  0.2002,
          0.0581, -0.3602, -1.1526,  0.0156,  0.2007,  0.0046, -0.1365,  1.2975,
          0.2322,  0.0982, -0.3282, -0.3466,  0.4115,  1.6366,  0.8127,  0.3331,
          0.2459,  0.7743, -1.3308,  0.5884, -0.5400]], device='cuda:0')
torques: [ 200.         -200.          200.         -187.49896192  200.
 -200.          -13.62115167  200.         -200.         -200.
  200.          123.56221806]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-1.7555,  1.9053,  0.0046, -1.0513,  0.2660,  1.3107, -0.4186, -0.5531,
         -0.5948,  0.0550, -1.5970, -1.1780]], device='cuda:0')
Scaled actions :  tensor([[-1.7555,  1.9053,  0.0046, -1.0513,  0.2660,  1.3107, -0.4186, -0.5531,
         -0.5948,  0.0550, -1.5970, -1.1780]], device='cuda:0')
obs :  tensor([[-0.0581, -0.1519, -0.4982, -0.0145, -0.0104, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0243, -0.1213,  0.0526,  0.1092, -0.6882, -0.0930,  0.0533,
          0.0952, -0.0205,  0.0327,  0.0166, -0.0695,  0.1696, -0.1961, -0.0352,
          0.3640,  1.5811,  0.7736,  0.5209,  0.1938,  0.0562, -0.2670,  1.2062,
         -1.1135, -1.7555,  1.9053,  0.0046, -1.0513,  0.2660,  1.3107, -0.4186,
         -0.5531, -0.5948,  0.0550, -1.5970, -1.1780]], device='cuda:0')
torques: [-143.7417634   -63.76293039 -200.          -94.47889995  200.
  200.         -200.         -139.29212099  200.         -200.
  -65.3330829   152.03821436]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.7979,  1.3432,  0.6550, -2.6668, -2.5118, -1.2633, -2.5435, -1.6437,
          0.4765,  1.2000,  0.3465,  1.4959]], device='cuda:0')
Scaled actions :  tensor([[ 0.7979,  1.3432,  0.6550, -2.6668, -2.5118, -1.2633, -2.5435, -1.6437,
          0.4765,  1.2000,  0.3465,  1.4959]], device='cuda:0')
obs :  tensor([[ 0.0865,  0.2429,  0.3302, -0.0115, -0.0110, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0217, -0.1620,  0.0328,  0.1490, -0.3578,  0.2324,  0.0718,
          0.1256, -0.0428,  0.0699, -0.1259, -0.3851, -0.5583, -0.2075, -0.1537,
          0.0634,  1.8219,  2.2212, -0.2294,  0.1548, -0.1540,  0.2925, -1.4067,
         -1.8902,  0.7979,  1.3432,  0.6550, -2.6668, -2.5118, -1.2633, -2.5435,
         -1.6437,  0.4765,  1.2000,  0.3465,  1.4959]], device='cuda:0')
torques: [-200.          200.          -67.71111613 -200.          200.
  200.         -200.         -200.         -200.           -6.59776987
 -200.         -200.        ]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 1.2143,  1.6082, -0.7653, -1.5730, -0.4677, -1.7269,  0.3922, -1.3832,
         -0.6336, -2.6539,  1.6753,  1.2703]], device='cuda:0')
Scaled actions :  tensor([[ 1.2143,  1.6082, -0.7653, -1.5730, -0.4677, -1.7269,  0.3922, -1.3832,
         -0.6336, -2.6539,  1.6753,  1.2703]], device='cuda:0')
obs :  tensor([[ 0.2899, -0.5285,  0.3345, -0.0188, -0.0189, -0.9996,  1.0000,  0.0000,
          0.0000, -0.0587, -0.1991,  0.0659,  0.1224, -0.2022,  0.4700, -0.0406,
          0.1445, -0.0413,  0.1422, -0.2048, -0.5515,  0.1255, -0.1655,  0.4331,
         -0.2848, -0.0814,  0.3485, -0.8383,  0.0289,  0.1452,  0.4230,  0.4376,
          0.0336,  1.2143,  1.6082, -0.7653, -1.5730, -0.4677, -1.7269,  0.3922,
         -1.3832, -0.6336, -2.6539,  1.6753,  1.2703]], device='cuda:0')
torques: [ 200.  200.  200. -200. -200. -200. -200. -200.  200.  200.  200.  200.]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.7584,  2.3084, -1.4312,  0.3542,  2.5644, -0.5951,  1.6621,  0.1420,
          0.7783, -2.7740, -0.8175,  0.5789]], device='cuda:0')
Scaled actions :  tensor([[-0.7584,  2.3084, -1.4312,  0.3542,  2.5644, -0.5951,  1.6621,  0.1420,
          0.7783, -2.7740, -0.8175,  0.5789]], device='cuda:0')
obs :  tensor([[ 0.2685,  0.2039, -0.5964, -0.0234, -0.0301, -0.9993,  1.0000,  0.0000,
          0.0000,  0.0349, -0.2134,  0.1427,  0.0508, -0.2818,  0.3319, -0.1047,
          0.1314, -0.0543,  0.2077,  0.0967, -0.3290,  0.7531,  0.0044,  0.3467,
         -0.4212, -0.3828, -1.5420,  0.0998, -0.1282, -0.2357,  0.2395,  2.3822,
          2.0020, -0.7584,  2.3084, -1.4312,  0.3542,  2.5644, -0.5951,  1.6621,
          0.1420,  0.7783, -2.7740, -0.8175,  0.5789]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.           11.99504695
 -200.          200.         -200.         -200.         -200.
  200.          200.        ]
データ収集:

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-1.8754,  1.5171,  0.4455,  1.0710,  2.1156,  2.5675, -0.5176, -0.9137,
         -0.3151, -0.0513, -3.2933, -1.2031]], device='cuda:0')
Scaled actions :  tensor([[-1.8754,  1.5171,  0.4455,  1.0710,  2.1156,  2.5675, -0.5176, -0.9137,
         -0.3151, -0.0513, -3.2933, -1.2031]], device='cuda:0')
obs :  tensor([[-0.4889,  0.2390, -0.3652, -0.0136, -0.0232, -0.9996,  1.0000,  0.0000,
          0.0000,  0.1049, -0.1823,  0.1630,  0.0091, -0.1467,  0.0207, -0.0294,
          0.1343, -0.0769,  0.2142,  0.3666, -0.0234,  0.0235,  0.2509, -0.0910,
         -0.0433,  1.5357, -1.2962,  0.6040,  0.0957, -0.0226, -0.1183,  0.5074,
          1.2652, -1.8754,  1.5171,  0.4455,  1.0710,  2.1156,  2.5675, -0.5176,
         -0.9137, -0.3151, -0.0513, -3.2933, -1.2031]], device='cuda:0')
torques: [-200.          200.         -200.          200.          200.
   66.99425303  200.          -63.92071664  200.         -200.
 -200.          -62.31427753]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.9067, -2.8569,  0.8072, -0.6059, -1.8895,  1.4190, -2.0980, -0.3961,
          0.6862,  1.0917,  0.1199, -0.4025]], device='cuda:0')
Scaled actions :  tensor([[ 0.9067, -2.8569,  0.8072, -0.6059, -1.8895,  1.4190, -2.0980, -0.3961,
          0.6862,  1.0917,  0.1199, -0.4025]], device='cuda:0')
obs :  tensor([[-0.4295,  0.2387,  0.4416, -0.0043, -0.0057, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0366, -0.1476,  0.0959,  0.0758,  0.0205,  0.0391,  0.0208,
          0.1418, -0.0823,  0.2010,  0.2025, -0.0279, -0.6130,  0.1579, -0.4256,
          0.4539,  1.2241,  0.9732, -0.0348,  0.0258,  0.0075, -0.0930, -1.6780,
         -0.9100,  0.9067, -2.8569,  0.8072, -0.6059, -1.8895,  1.4190, -2.0980,
         -0.3961,  0.6862,  1.0917,  0.1199, -0.4025]], device='cuda:0')
torques: [-200.  200.  200.  200.  200.  200. -200. -200. -200. -200. -200. -200.]
データ収集: step 11


In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 1.7143, -1.8478, -0.0910, -1.9593, -1.2690, -1.2503,  0.1751,  2.4953,
         -1.6506, -1.5860,  1.3302,  0.8977]], device='cuda:0')
Scaled actions :  tensor([[ 1.7143, -1.8478, -0.0910, -1.9593, -1.2690, -1.2503,  0.1751,  2.4953,
         -1.6506, -1.5860,  1.3302,  0.8977]], device='cuda:0')
obs :  tensor([[ 4.5108e-01, -5.5540e-01,  4.1487e-01, -1.1932e-02, -8.0100e-03,
         -9.9990e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -6.6420e-03,
         -1.5253e-01,  8.1309e-02,  1.1992e-01,  5.9203e-02,  3.8956e-01,
         -5.9826e-02,  1.1123e-01, -4.8207e-02,  1.9886e-01, -1.1122e-03,
         -1.6065e-01,  9.4429e-02, -1.7516e-01,  2.0995e-01,  3.2790e-02,
         -6.5368e-01,  2.1618e+00, -7.0368e-01, -3.0937e-01,  2.8787e-01,
          5.2842e-02, -1.9629e-01, -5.0605e-01,  1.7143e+00, -1.8478e+00,
         -9.1036e-02, -1.9593e+00, -1.2690e+00, -1.2503e+00,  1.7510e-01,
          2.4953e+00, -1.6506e+00, -1.5860e+00,  1.3302e+00,  8.

In [26]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=2.228, Scaled action max=2.228
Step 1/10, Total steps: 32
steps: 32
actions : tensor([[-0.4878,  1.7056, -0.7489,  0.8369, -3.2028,  0.6436, -0.1444,  2.2275,
          1.9721, -1.0972, -4.3128, -2.7630]], device='cuda:0')
target_dof_pos: tensor([[ 1.3871, -1.1847,  0.8960,  3.3602, -2.4817, -1.4051, -1.7818, -0.2685,
         -1.4334, -1.6893, -2.3825, -2.0675]], device='cuda:0')
Step 1: Original action max=1.642, Scaled action max=1.642
Step 2: Original action max=1.275, Scaled action max=1.275
データ収集完了: 10 steps collected with action_scale=1.0


In [54]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [55]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
